## Declarative pipelines — defining datasets in Python & SQL

Plain tables, views, MVs, and streaming tables are useful on their own. **Lakeflow Spark Declarative Pipelines** — formerly **Delta Live Tables (DLT)** — wrap them in a framework that adds: declarative dataset definitions (you describe *what* each dataset is; the framework builds the DAG and order), **expectations**, automatic **lineage** in Unity Catalog, CDC/SCD via `APPLY CHANGES INTO`, and managed compute + per-dataset retry.

**The rename watch:** *DLT* is the legacy name; *Lakeflow Spark Declarative Pipelines* is current — the exam may use either. The decorators are still `@dlt.table` / `@dlt.view`, and the SQL keywords are still `STREAMING TABLE` / `MATERIALIZED VIEW`.

**Python — decorator-based; each function returns a DataFrame:**

```python
import dlt
from pyspark.sql.functions import col

@dlt.table(comment="Raw card transactions")
def bronze_card_transactions():
    return spark.readStream.format("cloudFiles") \
        .option("cloudFiles.format", "json") \
        .load("/Volumes/fintech_dev/bronze/landing_zone/cards/")

@dlt.table(comment="Cleaned card transactions")
def silver_card_transactions():
    return dlt.read_stream("bronze_card_transactions") \
        .where(col("customer_id").isNotNull())
```

**SQL — same DAG, declarative:** `CREATE OR REFRESH STREAMING TABLE ... AS SELECT ... FROM STREAM(...)`.

**Two reader functions to memorise:**

- **`dlt.read("name")`** — read another dataset in the same pipeline as a **batch** snapshot.
- **`dlt.read_stream("name")`** — read another dataset in the same pipeline as a **stream**.

The framework wires the dependency graph from those reads — you never specify task order yourself.
